In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV , StratifiedKFold #***
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, f1_score, log_loss

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet, Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier, XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor, VotingClassifier, VotingRegressor

from tqdm import tqdm  # Provides the progress of model running
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
sonar = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\Sonar.csv")
X , y = sonar.drop(['Class'], axis=1), sonar['Class']
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=26)

In [3]:
lr = LogisticRegression()
params = {'C': np.linspace(0.001,20)}
gcv = GridSearchCV(estimator=lr, param_grid=params, cv=kfold, n_jobs=-1)
gcv.fit(X, y)
gcv.best_params_, gcv.best_score_

({'C': np.float64(3.266142857142857)}, np.float64(0.7737514518002324))

In [8]:
dtc = DecisionTreeClassifier(random_state=26)
lr = LogisticRegression()
rf = RandomForestClassifier(random_state=26)
gbm = XGBClassifier(random_state=26)

voting = VotingClassifier(estimators= [('TREE', dtc),('RF', rf), ('LR', lr),('XGB',gbm)], voting='soft')  
voting.get_params()

params = {'LR__C':np.linspace(0.001, 20, 10), 'TREE__max_depth':[3,5,6],
          'RF__max_features':[2,4,5], 'XGB__learning_rate':np.linspace(0.001, 0.4, 5)}

gcv = GridSearchCV(estimator=voting, param_grid=params, cv=kfold, n_jobs=-1)
gcv.fit(X, y)
gcv.best_params_, gcv.best_score_


({'LR__C': np.float64(15.555777777777775),
  'RF__max_features': 4,
  'TREE__max_depth': 3,
  'XGB__learning_rate': np.float64(0.2005)},
 np.float64(0.826945412311266))

In [9]:
print("Best Hyperparameters Found:")
print(gcv.best_params_)
print(f"\nBest Stratified 5-Fold CV Accuracy: {gcv.best_score_:.4f}")

Best Hyperparameters Found:
{'LR__C': np.float64(15.555777777777775), 'RF__max_features': 4, 'TREE__max_depth': 3, 'XGB__learning_rate': np.float64(0.2005)}

Best Stratified 5-Fold CV Accuracy: 0.8269


In [10]:
# Extracting the best voting classifier model
best_voting_clf = gcv.best_estimator_

In [7]:
# Predicting on the dataset to see final training metrics (or use X_test/y_test if split)
y_pred = best_voting_clf.predict(X)
print(f"Overall Accuracy on Data: {accuracy_score(y, y_pred):.4f}")

Overall Accuracy on Data: 0.9904
